
작업일: 2026.06.07

ORIG_SENT_CSV = Path(r"..\csv\original_csv\바른형태분석_세종_문어_형태분석_말뭉치_word_ver8.8_sen.csv")을 바탕으로 수정


기타: 인용문 처리에 문제가 있음. 

In [1]:
import pandas as pd
from datetime import datetime
from typing import Iterable, Optional, Tuple, List

# =========================================================
# Sentence CSV patcher (sen_id only) + modified_at (last only)
# - Only updates columns explicitly listed in UPDATE_COLS
# - Writes:
#   1) backup (original copy)
#   2) patched (updated CSV)
#   3) audit (optional; can be turned off)
# - Sets modified_at ONLY for rows that actually changed
# =========================================================

def safe_equal(a, b) -> bool:
    if pd.isna(a) and pd.isna(b):
        return True
    return a == b


def patch_sentence_csv_by_sen_id(
    orig: pd.DataFrame,
    corr: pd.DataFrame,
    backup_csv: str,
    patched_csv: str,
    *,
    update_cols: Iterable[str],
    encoding: str = "utf-8-sig",
    key_col: str = "sen_id",
    modified_col: str = "modified_at",
    now: Optional[str] = None,
    make_audit: bool = False,
    audit_csv: Optional[str] = None,
) -> Tuple[pd.DataFrame, Optional[pd.DataFrame]]:
    """
    Patch original sentence CSV with corrections CSV, matching by sen_id only.

    Parameters
    ----------
    orig_csv : str
        Original sentence CSV path.
    corr_csv : str
        Corrections CSV path.
    backup_csv : str
        Where to save a backup copy of the original.
    patched_csv : str
        Where to save the patched CSV.
    update_cols : Iterable[str]
        Columns that are allowed to be updated (whitelist).
        Only columns that exist in BOTH orig and corr will be applied.
    encoding : str
        CSV encoding (default utf-8-sig).
    key_col : str
        Matching key column (default sen_id).
    modified_col : str
        Column to store last modified timestamp (default modified_at).
    now : Optional[str]
        Timestamp string to write. If None, uses current local time.
    make_audit : bool
        If True, produces an audit dataframe & writes audit_csv.
    audit_csv : Optional[str]
        Audit CSV path. Required if make_audit=True.

    Returns
    -------
    patched_df : pd.DataFrame
    audit_df : Optional[pd.DataFrame]
    """
    if now is None:
        now = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

    # --- validate key ---
    if key_col not in orig.columns or key_col not in corr.columns:
        raise KeyError(f"Both orig and corr must contain key column: {key_col}")

    # --- backup ---
    orig.to_csv(backup_csv, index=False, encoding=encoding)

    # --- choose effective update columns (whitelist ∩ both files) ---
    update_cols = list(update_cols)
    effective_cols = [c for c in update_cols if c in orig.columns and c in corr.columns]
    if not effective_cols:
        raise ValueError(
            f"No effective columns to update. "
            f"Check update_cols and whether they exist in both files.\n"
            f"update_cols={update_cols}"
        )

    # --- normalize key type to string (safer) ---
    patched = orig.copy()
    patched[key_col] = patched[key_col].astype("string")

    corr_use = corr.copy()
    corr_use[key_col] = corr_use[key_col].astype("string")

    # --- corrections: drop NA key & keep last if duplicated key ---
    corr_use = corr_use.dropna(subset=[key_col]).drop_duplicates(subset=[key_col], keep="last")

    # --- ensure modified_col exists ---
    if modified_col not in patched.columns:
        patched[modified_col] = pd.NA

    # --- index by key ---
    patched = patched.set_index(key_col, drop=False)
    corr_use = corr_use.set_index(key_col, drop=False)

    common = patched.index.intersection(corr_use.index)
    print(f"matched rows: {len(common)}")
    print(f"update_cols (applied): {effective_cols}")

    audit_rows: List[dict] = []
    n_changed_cells = 0
    n_changed_rows = 0

    for sid in common:
        row_changed = False
        for col in effective_cols:
            old = patched.at[sid, col]
            new = corr_use.at[sid, col]

            if safe_equal(old, new):
                continue

            patched.at[sid, col] = new
            row_changed = True
            n_changed_cells += 1

            if make_audit:
                audit_rows.append({
                    key_col: sid,
                    "column": col,
                    "old": old,
                    "new": new,
                    "modified_at": now,
                })

        if row_changed:
            patched.at[sid, modified_col] = now
            n_changed_rows += 1

    patched = patched.reset_index(drop=True)
    patched.to_csv(patched_csv, index=False, encoding=encoding)

    print(f"changed rows : {n_changed_rows}")
    print(f"changed cells: {n_changed_cells}")
    print("backup ->", backup_csv)
    print("patched ->", patched_csv)

    audit_df = None
    if make_audit:
        if not audit_csv:
            raise ValueError("audit_csv is required when make_audit=True")
        audit_df = pd.DataFrame(audit_rows)
        audit_df.to_csv(audit_csv, index=False, encoding=encoding)
        print("audit  ->", audit_csv)

    return patched, audit_df

def pick_first_existing_column(df, candidates, *, required=True):
    """
    candidates 중 df에 실제로 존재하는 첫 번째 컬럼 이름을 반환.
    """
    for c in candidates:
        if c in df.columns:
            return c
    if required:
        raise KeyError(f"None of the candidate columns exist: {candidates}")
    return None

In [2]:
from pathlib import Path
from datetime import datetime

# 0) 경로 설정 (여기만 바꾸면 됨)
# ✅ 네 통합 원본 word CSV (한 개)
ORIG_SENT_CSV = Path(r"..\csv\original_csv\바른형태분석_세종_문어_형태분석_말뭉치_word_ver8.8_sen.csv")   # <-- 여기 바꿔

# ✅ 네가 검토 후 수정한 파일
CORRECTIONS_SENT_CSV = Path(r"..\csv\문장확인\문장확인_26.06.07.csv")   # 

# ✅ 출력 폴더
OUT_DIR = Path(r"../csv/sentence")
OUT_DIR.mkdir(parents=True, exist_ok=True)
stamp = datetime.now().strftime("%Y-%m-%d_%H-%M")
BACKUP_SENT_CSV = OUT_DIR / "_backup"/f"{ORIG_SENT_CSV.stem}__backup__{stamp}.csv"
PATCHED_SENT_CSV = OUT_DIR /"_patched"/ f"{ORIG_SENT_CSV.stem}__patched__{stamp}.csv"
AUDIT_SENT_CSV  = OUT_DIR / "patch_info"/f"{ORIG_SENT_CSV.stem}__audit__{stamp}.csv"


In [3]:
df_sen = pd.read_csv(ORIG_SENT_CSV, low_memory=False)
print(f"{ORIG_SENT_CSV}를 읽어왔습니다.")
# 3) sentence form 컬럼 자동 선택
sen_form_col = pick_first_existing_column(
        df_sen,
        ["sentence_form", "s_form", "form", "word_form", 'modified_at']
)
sen_use = (
        df_sen.rename(columns={sen_form_col: "sentence_form"})
    )

print(f"{sen_form_col}을 \"sentence_form\"으로 바꿨습니다.")

df_corr = pd.read_csv(CORRECTIONS_SENT_CSV)
print(f"{CORRECTIONS_SENT_CSV}를 읽어왔습니다.")


..\csv\original_csv\바른형태분석_세종_문어_형태분석_말뭉치_word_ver8.8_sen.csv를 읽어왔습니다.
sentence_form을 "sentence_form"으로 바꿨습니다.
..\csv\문장확인\문장확인_26.06.07.csv를 읽어왔습니다.


In [4]:
SEN_ID = "BA0225.00331"
print(df_sen.loc[df_sen["sen_id"] == SEN_ID, ["sentence_form"]])
df_corr.loc[df_corr["sen_id"] == SEN_ID, ["sen_id", "sentence_form"]]

                         sentence_form
232095  '…휴전선의 시계 청소작업은 새로운 단계로 접어들었다.


,sen_id,sentence_form
0,BA0225.00331,"""…휴전선의 시계 청소작업은 새로운 단계로 접어들었다."


In [5]:

# =========================
# 사용 예시
# =========================
UPDATE_COLS = {"sentence_form","review_flag", "review_note", "quote_type_fix"}
patched_df, _ = patch_sentence_csv_by_sen_id(
     orig=sen_use,
     corr=df_corr,
     backup_csv=BACKUP_SENT_CSV,
     patched_csv=PATCHED_SENT_CSV,
     update_cols=UPDATE_COLS,
     make_audit=True,              #
     audit_csv=AUDIT_SENT_CSV,    # make_audit=True일 때만 필요
)
print(f"{stamp}: 패치 완료.")

matched rows: 6
update_cols (applied): ['sentence_form']
changed rows : 2
changed cells: 2
backup -> ..\csv\sentence\_backup\바른형태분석_세종_문어_형태분석_말뭉치_word_ver8.8_sen__backup__2026-06-07_14-53.csv
patched -> ..\csv\sentence\_patched\바른형태분석_세종_문어_형태분석_말뭉치_word_ver8.8_sen__patched__2026-06-07_14-53.csv
audit  -> ..\csv\sentence\patch_info\바른형태분석_세종_문어_형태분석_말뭉치_word_ver8.8_sen__audit__2026-06-07_14-53.csv
2026-06-07_14-53: 패치 완료.
